# Day 46: Handling Imbalanced Data

## The Problem
Class imbalance shows up constantly in real ML problems - fraud detection, disease diagnosis, churn prediction - where the "interesting" class (fraud, disease, churn) is rare compared to the "normal" class. This is a problem because most classifiers are trained to minimize overall error rate, and if 90% of the data belongs to one class, a lazy model that always predicts that majority class already achieves 90% accuracy while being completely useless at its actual job of catching the minority cases. This means plain accuracy becomes a misleading metric on imbalanced data, and both the training process and the evaluation metrics need to be adjusted to account for it. The cell below builds a synthetic dataset with a 900:100 class split (90% class 0, 10% class 1) so we can see this problem play out directly in the next steps.

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(0)
n_majority, n_minority = 900, 100  # 90% vs 10% split - a realistic imbalance ratio
# majority class centered at (0,0), minority class shifted to (3,3) so the classes are still separable
X = np.vstack([np.random.randn(n_majority, 2), np.random.randn(n_minority, 2) + 3])
y = np.array([0] * n_majority + [1] * n_minority)
print('Class counts:', pd.Series(y).value_counts().to_dict())

## A Misleadingly High Accuracy
This cell trains a plain, unmodified logistic regression on the imbalanced data to demonstrate the problem concretely. Note `stratify=y` in the split - this ensures the train and test sets preserve the same 90/10 class ratio as the full dataset, which is standard practice for imbalanced data (a random split without stratification could accidentally leave very few minority samples in the test set). We then look at two metrics side by side: overall accuracy, and recall on the minority class specifically (recall = of all the actual minority-class samples, what fraction did the model correctly catch). The key thing to notice: accuracy can look deceptively good even while recall on the minority class is much lower, because the model can get most of its "correct" predictions just from the easy majority class - this gap is exactly why accuracy alone is the wrong metric to optimize for on imbalanced problems.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

# stratify=y keeps the 90/10 class ratio consistent between train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
model = LogisticRegression().fit(X_train, y_train)  # trained with no adjustment for imbalance
preds = model.predict(X_test)
print('Accuracy:', accuracy_score(y_test, preds))  # can look good even if minority class is poorly handled
print('Recall on minority class:', recall_score(y_test, preds))  # fraction of actual class-1 samples correctly caught

## Fix 1: Class Weights
One of the simplest fixes for imbalance is to change how mistakes are penalized during training rather than touching the data at all. Passing `class_weight='balanced'` tells scikit-learn to automatically weight each class inversely proportional to its frequency, so misclassifying a rare minority-class sample costs the loss function much more than misclassifying a common majority-class sample. This nudges the model's decision boundary to pay real attention to the minority class instead of largely ignoring it to optimize overall accuracy. It's an easy first thing to try because it requires no changes to the dataset itself - just one parameter. Compare the "Recall with class_weight" printed below to the plain model's minority-class recall from the previous cell; it should generally be higher, showing the model is now catching more of the minority-class cases (often at some cost to overall accuracy or precision, which is the expected tradeoff).

In [ ]:
weighted_model = LogisticRegression(class_weight='balanced').fit(X_train, y_train)  # auto-weights classes inversely to frequency
preds_w = weighted_model.predict(X_test)
print('Recall with class_weight:', recall_score(y_test, preds_w))  # compare to the unweighted model's recall above

## Fix 2: Oversampling the Minority Class
The other main strategy is to fix the imbalance in the data itself, before training, rather than adjusting weights inside the model. Oversampling duplicates existing minority-class rows (with replacement) until the classes are balanced (or close to it), so the model literally sees more minority examples during training and can't ignore them without hurting its training loss. This is simple and works with any model, but because it just repeats existing rows rather than creating new information, it can make a model more prone to overfitting on those repeated minority points. A more sophisticated alternative is SMOTE (Synthetic Minority Oversampling Technique) from the `imbalanced-learn` library, which generates new synthetic minority samples by interpolating between existing minority points instead of duplicating them - often a better choice in practice since it adds variety instead of just repeating rows. The code below does simple oversampling using scikit-learn's `resample` utility (only training data is resampled - the test set stays untouched and at its original imbalance, which matters because you always evaluate on realistic, unmodified data).

In [ ]:
from sklearn.utils import resample
import numpy as np

train_df = pd.DataFrame(X_train)
train_df['label'] = y_train
minority = train_df[train_df['label'] == 1]
majority = train_df[train_df['label'] == 0]
# sample the minority class WITH replacement up to the majority class size - i.e. duplicate rows to balance counts
minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
balanced_train = pd.concat([majority, minority_upsampled])  # combine into a class-balanced training set
print(balanced_train['label'].value_counts())